# 01 · Severson 2019 — load & explore

Sysblade Battery Digital Twin · W2 Day 1 morning.

**Goal.** Load the three Severson batch .mat files, sanity-check the data, extract the headline ΔQ₁₀₀₋₁₀(V) feature, and cache a clean Parquet for downstream notebooks (02 baseline, 03 LSTM).

**Pre-requisite.** `data/raw/severson/` must contain three .mat files. See `docs/severson_download.md` if they're missing — the next cell tells you which ones.

**Output.** `data/processed/severson_cells_features.parquet` with one row per cell, columns: `cell_id`, `batch`, `cycle_life`, `log_cycle_life`, `log_var_delta_q`, `policy`, `n_cycles_observed`.

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO / 'packages' / 'battery-twin'))

from data_loaders import severson_parser as sp
RAW = REPO / 'data' / 'raw' / 'severson'
PROCESSED = REPO / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
print('repo:', REPO)
print('raw :', RAW, '(exists)' if RAW.exists() else '(MISSING)')

## 1 · Verify the .mat files are present

In [ ]:
expected = [
    '2017-05-12_batchdata_updated_struct_errorcorrect.mat',
    '2017-06-30_batchdata_updated_struct_errorcorrect.mat',
    '2018-04-12_batchdata_updated_struct_errorcorrect.mat',
]
for f in expected:
    p = RAW / f
    if p.exists():
        print(f'  ok  {p.name}  ({p.stat().st_size/1e9:.2f} GB)')
    else:
        print(f'  MISSING  {p.name}')
        print(f'           → see docs/severson_download.md')

## 2 · Load all three batches

First run takes ~2-5 minutes (parsing 6 GB). Subsequent runs reload from a cached pickle so you don't pay the cost twice.

In [ ]:
import pickle
CACHE = PROCESSED / 'severson_cells.pkl'

if CACHE.exists():
    print(f'reloading cached parse from {CACHE}')
    cells = pickle.loads(CACHE.read_bytes())
else:
    t0 = time.time()
    cells = sp.load_all(RAW)
    print(f'parsed {len(cells)} cells in {time.time()-t0:.1f}s')
    CACHE.write_bytes(pickle.dumps(cells))
    print(f'cached → {CACHE}')
print()
print(f'cells per batch: ' + ', '.join(
    f'{b}={sum(1 for c in cells if c.batch==b)}' for b in ('b1','b2','b3')
))

## 3 · Summary statistics

Severson 2019 had 124 cells total. Some have very few cycles (early failures), some have 1000+. Cycle-life distribution is heavy-right-tailed.

In [ ]:
df = pd.DataFrame([{
    'cell_id': c.cell_id,
    'batch':   c.batch,
    'cycle_life': c.cycle_life,
    'n_cycles_observed': c.n_cycles,
    'policy':  c.policy,
} for c in cells])
print(df.describe(percentiles=[.1,.25,.5,.75,.9]).round(0))
print()
print('cycle life buckets:')
print(pd.cut(df['cycle_life'], bins=[0,300,600,1000,1500,3000]).value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['cycle_life'], bins=30, edgecolor='k', alpha=0.7)
axes[0].set_xlabel('Cycle life (cycles to 80% SOH)'); axes[0].set_ylabel('count')
axes[0].set_title('Severson cycle-life distribution')
axes[0].grid(alpha=0.3)
for batch, color in zip(['b1','b2','b3'], ['C0','C1','C2']):
    sub = df[df.batch == batch]
    axes[1].scatter(sub.index, sub['cycle_life'], label=f'{batch} (n={len(sub)})', alpha=0.7, color=color)
axes[1].set_xlabel('cell index'); axes[1].set_ylabel('cycle life')
axes[1].set_title('Cycle life by batch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4 · Eyeball one cell — discharge Q-V curves at cycles 10, 100, end

We pick a cell with > 500 observed cycles to see the shift between early-life and late-life curves. The discrepancy between cycle 10 and cycle 100 is what the headline ΔQ₁₀₀₋₁₀(V) feature captures.

In [ ]:
cell = next((c for c in cells if c.n_cycles > 500), cells[0])
print(f'inspecting {cell.cell_id}, cycle_life={cell.cycle_life}, n_cycles={cell.n_cycles}')

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
for idx, label, color in [(10, 'cycle 10', 'C0'),
                          (100, 'cycle 100', 'C1'),
                          (cell.n_cycles, f'cycle {cell.n_cycles} (last)', 'C3')]:
    try:
        c = cell.cycle(idx)
        ax.plot(c.V, c.Qd, lw=1.4, label=label, color=color)
    except IndexError:
        pass
ax.set_xlabel('Voltage (V)'); ax.set_ylabel('Discharge Q (Ah)')
ax.set_title(f'{cell.cell_id} · discharge Q-V curves'); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 5 · Compute ΔQ₁₀₀₋₁₀(V) for that cell

Severson's headline finding: **the *variance* of this curve, alone, predicts cycle life to within ~9 % across the whole dataset.** Let's plot it.

In [ ]:
voltage_grid = np.linspace(2.0, 3.5, 1000)
delta = sp.delta_q_100_10(cell, voltage_grid)
if delta is None:
    print('cell does not have cycles 10 and 100')
else:
    plt.figure(figsize=(8, 4))
    plt.plot(voltage_grid, delta * 1000, lw=1.4)  # mAh for readability
    plt.xlabel('Voltage (V)'); plt.ylabel('ΔQ₁₀₀₋₁₀ (mAh)')
    plt.title(f'{cell.cell_id} · ΔQ between cycle 100 and cycle 10')
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
    valid = delta[np.isfinite(delta)]
    print(f'variance(ΔQ) = {np.var(valid):.6e} (Ah²)')
    print(f'log10(variance) = {np.log10(np.var(valid)):.3f}')

## 6 · Compute the feature for every cell

This is the input we'll feed into the Severson baseline regression in notebook `02_severson_baseline.ipynb`.

In [ ]:
rows = sp.features_for_all(cells)
feat_df = pd.DataFrame(rows)
print(f'{len(feat_df)} cells with valid ΔQ feature  (out of {len(cells)} parsed)')
print(feat_df.head(10))

## 7 · The money plot

This is the scatter that the Severson 2019 paper used as its headline. If it shows a clean linear-ish trend, the simple regression in notebook 02 will hit ~9 % MAPE.

In [ ]:
plt.figure(figsize=(8, 5))
for batch, color in zip(['b1','b2','b3'], ['C0','C1','C2']):
    sub = feat_df[feat_df.batch == batch]
    plt.scatter(sub.log_var_delta_q, sub.log_cycle_life,
                label=f'{batch} (n={len(sub)})', alpha=0.7, color=color)
plt.xlabel('log₁₀(var(ΔQ₁₀₀₋₁₀(V)))'); plt.ylabel('log₁₀(cycle life)')
plt.title('Severson 2019 headline relationship · should be roughly linear')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

# Pearson correlation in log space
r = np.corrcoef(feat_df.log_var_delta_q, feat_df.log_cycle_life)[0, 1]
print(f'Pearson r in log-log space: {r:.3f}  (Severson paper ≈ -0.93)')

## 8 · Cache features

Persist to Parquet so notebook 02 reloads in ~50 ms instead of re-parsing 6 GB of .mat.

In [ ]:
out = PROCESSED / 'severson_cells_features.parquet'
feat_df.to_parquet(out, index=False)
print(f'wrote {out} ({out.stat().st_size/1024:.1f} KiB)')

## Pass criteria

- [ ] All three .mat files load without errors
- [ ] ≥ 100 cells parsed (Severson reports 124, expect within ±5)
- [ ] ΔQ for one example cell looks like a smooth curve, not noise
- [ ] Pearson r in log-log space is more negative than −0.85 (paper ≈ −0.93)
- [ ] Parquet feature file written to `data/processed/`

If all four pass: notebook 02 baseline regression should hit < 12 % MAPE on a simple holdout split, and < 10 % is in reach with the same train/test split Severson used (b1+b2 train, b3 test).